In [1]:
#conda install geopandas

In [2]:
import pandas as pd
import quality_func

import geopandas as gpd

In [3]:
zone = quality_func.load_zone_data()
zone.shape

Set with labels : (265, 3)


(265, 3)

In [4]:
zone.head(5)

,LocationID,Borough,Zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


In [5]:
files = ["data/uber-trip-data/uber-raw-data-apr14.csv", "data/uber-trip-data/uber-raw-data-may14.csv", "data/uber-trip-data/uber-raw-data-jun14.csv", 'data/uber-trip-data/uber-raw-data-jul14.csv', 'data/uber-trip-data/uber-raw-data-aug14.csv', 'data/uber-trip-data/uber-raw-data-sep14.csv']

raw_2014 = quality_func.load_2014_data(files)
raw_2014.shape

Set with labels : (4534327, 4)


(4534327, 4)

In [6]:
raw_2014.head(5)

,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [7]:
raw_2015 = quality_func.load_2015_data('data/uber-trip-data/uber-raw-data-janjune-15.csv')

Set with labels : (14270479, 4)


# Explore dataset

In [8]:
dataframes_viz = [zone, raw_2014, raw_2015]
[print(f"Display of dataset: {display(df.head())}") for df in dataframes_viz]

,LocationID,Borough,Zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


Display of dataset: None


,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


Display of dataset: None


,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
0,B02617,2015-05-17 09:47:00,B02617,141
1,B02617,2015-05-17 09:47:00,B02617,65
2,B02617,2015-05-17 09:47:00,B02617,100
3,B02617,2015-05-17 09:47:00,B02774,80
4,B02617,2015-05-17 09:47:00,B02617,90


Display of dataset: None


[None, None, None]

In [9]:
zones_official = gpd.read_file("data/taxi_zones/taxi_zones/taxi_zones.shp")[["LocationID", "geometry"]].to_crs("EPSG:4326")

In [10]:
loc_2014 = gpd.GeoDataFrame(
    raw_2014,
    geometry=gpd.points_from_xy(raw_2014["Lon"], raw_2014["Lat"]),
    crs="EPSG:4326"
)

In [11]:
dataset_2014 = gpd.sjoin(loc_2014, zones_official, how="left", predicate="within").drop(columns=["index_right"])

In [12]:
dataset_2014.head(5)

,Date/Time,Lat,Lon,Base,geometry,LocationID
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512,POINT (-73.9549 40.769),140.0
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512,POINT (-74.0345 40.7267),NaN
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512,POINT (-73.9873 40.7316),79.0
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512,POINT (-73.9776 40.7588),161.0
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512,POINT (-73.9722 40.7594),162.0


In [13]:
zones_official["rep_pt"] = zones_official.geometry.representative_point()
zones_official["lon"] = zones_official["rep_pt"].x
zones_official["lat"] = zones_official["rep_pt"].y

In [14]:
raw_2015 = raw_2015.rename(columns={"locationID": "LocationID"})

In [15]:
dataset_2015 = raw_2015.merge(zones_official[["LocationID", "lat", "lon"]], on="LocationID", how="left")

In [16]:
dataset_2015.head()

,Dispatching_base_num,Pickup_date,Affiliated_base_num,LocationID,lat,lon
0,B02617,2015-05-17 09:47:00,B02617,141,40.766838,-73.959712
1,B02617,2015-05-17 09:47:00,B02617,65,40.695371,-73.985571
2,B02617,2015-05-17 09:47:00,B02617,100,40.753513,-73.988786
3,B02617,2015-05-17 09:47:00,B02774,80,40.714461,-73.942336
4,B02617,2015-05-17 09:47:00,B02617,90,40.742546,-73.996777


In [17]:
display(zone.head(1),zones_official.head(1) ,dataset_2014.head(1), dataset_2015.head(1))

,LocationID,Borough,Zone
0,1,EWR,Newark Airport


,LocationID,geometry,rep_pt,lon,lat
0,1,"POLYGON ((-74.18445 40.695, -74.18449 40.69509...",POINT (-74.17678 40.68951),-74.176778,40.689515


,Date/Time,Lat,Lon,Base,geometry,LocationID
0,4/1/2014 0:11:00,40.769,-73.9549,B02512,POINT (-73.9549 40.769),140.0


,Dispatching_base_num,Pickup_date,Affiliated_base_num,LocationID,lat,lon
0,B02617,2015-05-17 09:47:00,B02617,141,40.766838,-73.959712


In [19]:
dataset_zone = pd.merge(zone, zones_official[["LocationID", "lat", "lon"]], on="LocationID", how="left")

In [20]:
dataset_2014 = dataset_2014.drop(columns=["geometry"])

In [21]:
display(dataset_zone.head(1), dataset_2014.head(1), dataset_2015.head(1))

,LocationID,Borough,Zone,lat,lon
0,1,EWR,Newark Airport,40.689515,-74.176778


,Date/Time,Lat,Lon,Base,LocationID
0,4/1/2014 0:11:00,40.769,-73.9549,B02512,140.0


,Dispatching_base_num,Pickup_date,Affiliated_base_num,LocationID,lat,lon
0,B02617,2015-05-17 09:47:00,B02617,141,40.766838,-73.959712
